# Cell Theory and the Basic Unit of Life Workflow

This notebook scaffold supports the article **Cell Theory and the Basic Unit of Life**. It can be expanded with cell-growth modeling, viability decay, membrane flux, cell-cycle compartments, imaging features, condition scoring, and provenance notes.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

article_dir = Path.cwd().parent
counts = pd.read_csv(article_dir / 'data' / 'cell_counts.csv')
rows = []
for condition, group in counts.groupby('condition'):
    slope, intercept = np.polyfit(group['time_h'], np.log(group['cells']), 1)
    rows.append({'condition': condition, 'growth_rate_per_h': slope, 'N0': np.exp(intercept), 'doubling_time_h': np.log(2) / slope})
pd.DataFrame(rows).round(5)

In [ ]:
viability = pd.read_csv(article_dir / 'data' / 'viability_observations.csv')
rows = []
for condition, group in viability.groupby('condition'):
    slope, intercept = np.polyfit(group['time_h'], np.log(group['viable_cells']), 1)
    k = -slope
    rows.append({'condition': condition, 'loss_rate_per_h': k, 'L0': np.exp(intercept), 'half_life_h': np.log(2) / k})
pd.DataFrame(rows).round(5)

In [ ]:
flux = pd.read_csv(article_dir / 'data' / 'membrane_gradients.csv')
flux['concentration_gradient'] = (flux['concentration_outside'] - flux['concentration_inside']) / flux['distance_cm']
flux['membrane_flux'] = -flux['diffusion_coefficient_cm2_s'] * flux['concentration_gradient']
flux.round(8)

In [ ]:
condition = pd.read_csv(article_dir / 'data' / 'cell_condition_sites.csv')
condition['cell_condition_score'] = (
    0.18 * condition['membrane_integrity'] +
    0.22 * condition['metabolic_activity'] +
    0.18 * condition['proliferation_capacity'] +
    0.17 * condition['genomic_stability'] +
    0.15 * condition['organelle_function'] +
    0.10 * (1 - condition['stress_penalty'])
)
condition.sort_values('cell_condition_score', ascending=False).round(3)